# Graph Feature Extraction — BankSim

Build bipartite customer–merchant graph **từ train set only**, extract graph features, concat vào existing feature set.

Features được thêm:
- `customer_degree` — số merchant unique customer đã giao dịch
- `merchant_degree` — số customer unique merchant đã giao dịch
- `merchant_fraud_rate` — tỉ lệ fraud của merchant trong train
- `customer_fraud_rate` — tỉ lệ fraud của customer trong train
- `customer_pagerank` — PageRank score của customer node
- `merchant_pagerank` — PageRank score của merchant node

> **No data leakage**: graph chỉ build từ X_train + y_train.

In [1]:
from pathlib import Path
ROOT = Path("../..").resolve()

import numpy as np
import pandas as pd
import networkx as nx
import joblib
import os

DATA_DIR = ROOT / 'src/data/processed'

# Install networkx nếu chưa có
try:
    import networkx
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'networkx'])
    import networkx as nx

print('NetworkX version:', nx.__version__)

NetworkX version: 3.5


## Step 1 — Load Data

In [2]:
X_train     = joblib.load(os.path.join(DATA_DIR, 'X_train.pkl'))
y_train     = joblib.load(os.path.join(DATA_DIR, 'y_train.pkl'))
X_val       = joblib.load(os.path.join(DATA_DIR, 'X_val.pkl'))
y_val       = joblib.load(os.path.join(DATA_DIR, 'y_val.pkl'))
X_test      = joblib.load(os.path.join(DATA_DIR, 'X_test.pkl'))
y_test      = joblib.load(os.path.join(DATA_DIR, 'y_test.pkl'))
X_train_res = joblib.load(os.path.join(DATA_DIR, 'X_train_res.pkl'))
y_train_res = joblib.load(os.path.join(DATA_DIR, 'y_train_res.pkl'))
feature_cols = joblib.load(os.path.join(DATA_DIR, 'feature_cols.pkl'))

print('feature_cols:', feature_cols)
print(f'\nShapes before adding graph features:')
print(f'  X_train     : {X_train.shape}')
print(f'  X_val       : {X_val.shape}')
print(f'  X_test      : {X_test.shape}')
print(f'  X_train_res : {X_train_res.shape}')

# Column indices (0-based)
CUST_IDX = feature_cols.index('customer')
MERCH_IDX = feature_cols.index('merchant')
print(f'\ncustomer col idx : {CUST_IDX}')
print(f'merchant col idx : {MERCH_IDX}')

feature_cols: ['step', 'customer', 'age', 'gender', 'merchant', 'category', 'amount', 'age_unknown', 'amount_log', 'amount_zscore', 'cust_mean', 'cust_std', 'cust_max', 'cust_tx_count', 'amount_deviation', 'amount_to_max_ratio', 'category_risk', 'merchant_risk', 'is_new_merchant', 'is_new_category', 'is_round_amount', 'is_micro_tx', 'hour', 'is_night', 'is_weekend', 'age_group_mean', 'age_group_std', 'age_group_deviation']

Shapes before adding graph features:
  X_train     : (416250, 28)
  X_val       : (89196, 28)
  X_test      : (89197, 28)
  X_train_res : (452082, 28)

customer col idx : 1
merchant col idx : 4


## Step 2 — Build Bipartite Graph từ Train Set

In [3]:
# Chuyển sang DataFrame để tiện xử lý
train_df = pd.DataFrame(X_train, columns=feature_cols)
train_df['fraud'] = y_train

# Encode customer/merchant thành prefix string để phân biệt node type trong graph
train_df['cust_node']  = 'C_' + train_df['customer'].astype(int).astype(str)
train_df['merch_node'] = 'M_' + train_df['merchant'].astype(int).astype(str)

# Build undirected bipartite graph
G = nx.Graph()

# Add nodes với bipartite attribute
cust_nodes  = train_df['cust_node'].unique()
merch_nodes = train_df['merch_node'].unique()
G.add_nodes_from(cust_nodes,  bipartite=0)  # customer side
G.add_nodes_from(merch_nodes, bipartite=1)  # merchant side

# Add edges (một transaction = một edge, có thể có bội số — dùng add_edges_from)
edges = list(zip(train_df['cust_node'], train_df['merch_node']))
G.add_edges_from(edges)

print(f'Graph nodes  : {G.number_of_nodes()} ({len(cust_nodes)} customers + {len(merch_nodes)} merchants)')
print(f'Graph edges  : {G.number_of_edges()} (unique customer-merchant pairs)')

Graph nodes  : 4161 (4111 customers + 50 merchants)
Graph edges  : 38924 (unique customer-merchant pairs)


## Step 3 — Tính Graph Features

In [4]:
# -- Degree features --
# customer_degree = số merchant unique (neighbors trong bipartite graph)
# merchant_degree = số customer unique
degree_dict = dict(G.degree())

customer_degree = {node: degree_dict[node] for node in cust_nodes}
merchant_degree = {node: degree_dict[node] for node in merch_nodes}

# -- Fraud rate features --
cust_fraud = (
    train_df.groupby('cust_node')['fraud']
    .mean()
    .to_dict()
)
merch_fraud = (
    train_df.groupby('merch_node')['fraud']
    .mean()
    .to_dict()
)

# -- PageRank --
print('Computing PageRank (may take a few seconds)...')
pagerank = nx.pagerank(G, alpha=0.85, max_iter=100, tol=1e-6)
customer_pagerank = {node: pagerank[node] for node in cust_nodes}
merchant_pagerank = {node: pagerank[node] for node in merch_nodes}

print('Done.')
print(f'  Sample customer_degree : {list(customer_degree.items())[:3]}')
print(f'  Sample merch_fraud_rate: {list(merch_fraud.items())[:3]}')
print(f'  Sample customer_pr     : {list(customer_pagerank.items())[:3]}')

Computing PageRank (may take a few seconds)...
Done.
  Sample customer_degree : [('C_3944', 6), ('C_879', 7), ('C_950', 7)]
  Sample merch_fraud_rate: [('M_0', 0.0), ('M_1', 0.0), ('M_10', 0.06751592356687898)]
  Sample customer_pr     : [('C_3944', 9.686185722653592e-05), ('C_879', 0.00010686458837509525), ('C_950', 0.00010646492782966809)]


## Step 4 — Global Mean Fallback (dùng cho val/test unseen nodes)

In [5]:
global_means = {
    'customer_degree'     : np.mean(list(customer_degree.values())),
    'merchant_degree'     : np.mean(list(merchant_degree.values())),
    'merchant_fraud_rate' : np.mean(list(merch_fraud.values())),
    'customer_fraud_rate' : np.mean(list(cust_fraud.values())),
    'customer_pagerank'   : np.mean(list(customer_pagerank.values())),
    'merchant_pagerank'   : np.mean(list(merchant_pagerank.values())),
}

print('Global mean fallback values:')
for k, v in global_means.items():
    print(f'  {k:25s}: {v:.6f}')

Global mean fallback values:
  customer_degree          : 9.468256
  merchant_degree          : 778.480000
  merchant_fraud_rate      : 0.223325
  customer_fraud_rate      : 0.028293
  customer_pagerank        : 0.000131
  merchant_pagerank        : 0.009190


## Step 5 — Hàm extract_graph_features

In [6]:
def extract_graph_features(X: np.ndarray, cust_idx: int, merch_idx: int) -> np.ndarray:
    """
    Trả về array shape (n_samples, 6) chứa 6 graph features.
    customer/merchant values được round về int để lookup trong train graph.
    Unseen nodes -> global mean của train.
    """
    n = len(X)
    feat = np.zeros((n, 6), dtype=np.float64)

    for i in range(n):
        cust_raw  = int(round(X[i, cust_idx]))
        merch_raw = int(round(X[i, merch_idx]))
        cnode = f'C_{cust_raw}'
        mnode = f'M_{merch_raw}'

        feat[i, 0] = customer_degree.get(cnode,     global_means['customer_degree'])
        feat[i, 1] = merchant_degree.get(mnode,     global_means['merchant_degree'])
        feat[i, 2] = merch_fraud.get(mnode,         global_means['merchant_fraud_rate'])
        feat[i, 3] = cust_fraud.get(cnode,          global_means['customer_fraud_rate'])
        feat[i, 4] = customer_pagerank.get(cnode,   global_means['customer_pagerank'])
        feat[i, 5] = merchant_pagerank.get(mnode,   global_means['merchant_pagerank'])

    return feat


GRAPH_FEAT_NAMES = [
    'customer_degree',
    'merchant_degree',
    'merchant_fraud_rate',
    'customer_fraud_rate',
    'customer_pagerank',
    'merchant_pagerank',
]

print('extract_graph_features ready.')

extract_graph_features ready.


## Step 6 — Extract & Concat cho tất cả splits

In [7]:
splits = {
    'X_train'     : X_train,
    'X_val'       : X_val,
    'X_test'      : X_test,
    'X_train_res' : X_train_res,
}

results = {}
for name, X in splits.items():
    print(f'Extracting graph features for {name} ...', end=' ')
    gf = extract_graph_features(X, CUST_IDX, MERCH_IDX)
    X_new = np.hstack([X, gf])
    results[name] = X_new
    print(f'{X.shape} → {X_new.shape}')

feature_cols_graph = feature_cols + GRAPH_FEAT_NAMES
print(f'\nTotal features: {len(feature_cols_graph)}')
print('New features:', GRAPH_FEAT_NAMES)

Extracting graph features for X_train ... (416250, 28) → (416250, 34)
Extracting graph features for X_val ... (89196, 28) → (89196, 34)
Extracting graph features for X_test ... (89197, 28) → (89197, 34)
Extracting graph features for X_train_res ... (452082, 28) → (452082, 34)

Total features: 34
New features: ['customer_degree', 'merchant_degree', 'merchant_fraud_rate', 'customer_fraud_rate', 'customer_pagerank', 'merchant_pagerank']


## Step 7 — Kiểm tra NaN / phân phối

In [8]:
for name, X_new in results.items():
    df_check = pd.DataFrame(X_new[:, -6:], columns=GRAPH_FEAT_NAMES)
    nan_count = df_check.isnull().sum().sum()
    print(f'{name}: NaN={nan_count}')
    print(df_check.describe().to_string())
    print()

X_train: NaN=0
       customer_degree  merchant_degree  merchant_fraud_rate  customer_fraud_rate  customer_pagerank  merchant_pagerank
count    416250.000000    416250.000000        416250.000000        416250.000000      416250.000000      416250.000000
mean          9.539073      3202.587962             0.012651             0.012651           0.000132           0.038088
std           2.759780       717.523919             0.080338             0.062935           0.000028           0.008671
min           2.000000         2.000000             0.000000             0.000000           0.000056           0.000061
25%           8.000000      3173.000000             0.000000             0.000000           0.000117           0.037493
50%           9.000000      3173.000000             0.000000             0.000000           0.000127           0.037493
75%          11.000000      3732.000000             0.000000             0.000000           0.000147           0.044710
max          28.000000   

## Step 8 — Save

In [9]:
save_map = {
    'X_train_graph'     : results['X_train'],
    'X_val_graph'       : results['X_val'],
    'X_test_graph'      : results['X_test'],
    'X_train_res_graph' : results['X_train_res'],
    'feature_cols_graph': feature_cols_graph,
}

for fname, obj in save_map.items():
    out_path = os.path.join(DATA_DIR, f'{fname}.pkl')
    joblib.dump(obj, out_path)
    kb = os.path.getsize(out_path) / 1024
    print(f'Saved {fname}.pkl  ({kb:.0f} KB)')

print('\nDone!')

Saved X_train_graph.pkl  (110567 KB)
Saved X_val_graph.pkl  (23693 KB)
Saved X_test_graph.pkl  (23693 KB)
Saved X_train_res_graph.pkl  (120085 KB)
Saved feature_cols_graph.pkl  (1 KB)

Done!


## Step 9 — Quick sanity: AUC-PR với LightGBM

In [10]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score

X_tr_g  = results['X_train_res']
X_val_g = results['X_val']

# Baseline (without graph features)
clf_base = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
                               random_state=42, verbose=-1, n_jobs=-1)
clf_base.fit(X_train_res, y_train_res)
ap_base = average_precision_score(y_val, clf_base.predict_proba(X_val)[:, 1])

# With graph features
clf_graph = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63,
                                random_state=42, verbose=-1, n_jobs=-1)
clf_graph.fit(X_tr_g, y_train_res)
ap_graph = average_precision_score(y_val, clf_graph.predict_proba(X_val_g)[:, 1])

print(f'AUC-PR without graph features : {ap_base:.4f}')
print(f'AUC-PR with    graph features : {ap_graph:.4f}')
print(f'Delta                         : {ap_graph - ap_base:+.4f}')

c:\Users\GIGABYTE\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


AUC-PR without graph features : 0.9365
AUC-PR with    graph features : 0.9296
Delta                         : -0.0068


c:\Users\GIGABYTE\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 10. Overlay PR-AUC Curve (All Models)
Kết quả từ Retrain_Graph_ErrorAnalysis.ipynb

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
import joblib
import os

graph_probs = joblib.load(os.path.join(DATA_DIR, 'graph_test_probs_retrain.pkl'))
base_probs = joblib.load(os.path.join(DATA_DIR, 'base_probs_retrain.pkl'))

plt.figure(figsize=(10, 8))
colors = {'BRF': 'green', 'LightGBM': 'blue', 'XGBoost': 'red', 'CatBoost': 'purple'}

for name in graph_probs.keys():
    # Baseline PR-AUC
    prec_b, rec_b, _ = precision_recall_curve(y_test, base_probs[name])
    auc_b = average_precision_score(y_test, base_probs[name])
    plt.plot(rec_b, prec_b, linestyle='--', color=colors[name], alpha=0.5,
             label=f'{name} Baseline (PR-AUC={auc_b:.4f})')
    
    # Graph PR-AUC
    prec_g, rec_g, _ = precision_recall_curve(y_test, graph_probs[name])
    auc_g = average_precision_score(y_test, graph_probs[name])
    plt.plot(rec_g, prec_g, linestyle='-', color=colors[name],
             label=f'{name} Graph (PR-AUC={auc_g:.4f})')

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('PR-AUC Curve: Baseline vs Graph-Enhanced Models (Test Set)')
plt.legend(fontsize=9, loc='lower left')
plt.grid(True)
plt.tight_layout()
plt.show()
